# Notebook 02: Web Scraping & Text Extraction

**Time:** 30 minutes  
**Prerequisites:** Notebook 01 complete  
**Goal:** Extract clean text from web pages using traditional and modern tools

This notebook will:
1. Extract text from web pages with trafilatura (traditional approach)
2. Use Crawl4AI for LLM-ready markdown extraction (modern 2025 approach)
3. Scrape arXiv paper abstracts for a pretraining dataset
4. Compare extraction quality between tools

> **Why this matters:** The quality of LLM pretraining data starts with extraction. Garbage in, garbage out. Modern tools like Crawl4AI (50K+ GitHub stars) produce cleaner, LLM-optimized markdown directly, while trafilatura remains a solid baseline for simpler extraction tasks.

In [26]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

# Week 3 specific imports
import src.scraping_utils
importlib.reload(src.scraping_utils)
from src.scraping_utils import (
    extract_with_trafilatura,
    scrape_arxiv_abstracts,
    compare_extractors,
    extract_with_crawl4ai   
)

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("Setup complete -- ready for Notebook 02")

✓ Gemini client initialized (gemini-3-flash-preview)
Setup complete -- ready for Notebook 02


---

## Part 1: Common Crawl & trafilatura

Common Crawl is a massive public archive of web pages -- like a snapshot of the entire internet, updated monthly. It's used to train models like GPT, Claude, and LLaMA.

**trafilatura** is a Python library that extracts the main content from web pages, stripping away navigation, ads, and boilerplate. Think of it as using a highlighter on millions of web pages to extract just the important sentences.

In [3]:
print("=" * 65)
print("Experiment 1: trafilatura Text Extraction")
print("=" * 65)
print()

# Extract text from an arXiv abstract page
url = "https://arxiv.org/abs/2404.00001"
result = extract_with_trafilatura(url)

print(f"\nExtracted text preview:")
print(result['text'][:500] if result['text'] else 'No text extracted')

Experiment 1: trafilatura Text Extraction

[trafilatura] Fetching: https://arxiv.org/abs/2404.00001
  Extracted 2,389 chars in 0.1s

Extracted text preview:
Physics > Physics Education
[Submitted on 3 Feb 2024]
Title:Uso de herramientas digitales matemáticas en la Educación Secundaria
View PDF HTML (experimental)Abstract:Information and Community Technologies (ICT) are very present in our society nowadays and particularly in the educative field. In just two decades, we have passed from a learning based, in many cases, on the master lessons to one such that methodologies like the flipped classroom or the gamification are stronger than ever. Along thi


In [4]:
print("=" * 65)
print("Experiment 2: Extract from a Blog Post")
print("=" * 65)
print()

# Try a blog/news article
blog_url = "https://ai.meta.com/blog/meta-llama-4/"
blog_result = extract_with_trafilatura(blog_url)

print(f"\nExtracted {blog_result['char_count']:,} chars")
print(f"Preview: {blog_result['text'][:400]}..." if blog_result['text'] else 'No text extracted')

Experiment 2: Extract from a Blog Post

[trafilatura] Fetching: https://ai.meta.com/blog/meta-llama-4/
  Extracted 120 chars in 1.6s

Extracted 120 chars
Preview: You might find what you need on our homepage .
Foundational models
Our approach
Research
Meta AI
Latest news
Meta © 2026...


In [11]:
# TODO 1: Extract text from a URL of your choice
#
# Pick a web page related to your interests (blog post, docs page, wiki article).
# Extract the text and evaluate the quality.

# my_url = "[STUDENT: PASTE YOUR URL HERE]"
my_url = "https://github.com/openai/whisper"#"https://github.com/facebookresearch/tribev2"

print("=" * 65)
print("TODO 1: Custom URL Extraction")
print("=" * 65)
print()

my_result = extract_with_trafilatura(my_url)
print(f"\nExtracted {my_result['char_count']:,} chars")
print(f"Preview: {my_result['text'][:300]}..." if my_result['text'] else 'No text extracted')


TODO 1: Custom URL Extraction

[trafilatura] Fetching: https://github.com/openai/whisper
  Extracted 6,209 chars in 0.9s

Extracted 6,209 chars
Preview: [Blog] [Paper] [Model card] [Colab example]
Whisper is a general-purpose speech recognition model. It is trained on a large dataset of diverse audio and is also a multitasking model that can perform multilingual speech recognition, speech translation, and language identification.
A Transformer seque...


In [12]:

todo1_reflection = """


- What URL did you choose and why?
https://github.com/openai/whisper
Because in my project, I want to use the whisper model to transcribe audio, 
so i want to see how to use Whisper and what are the features of it. 
I also want to see how well trafilatura can extract the content from a github page, which is different from a typical blog post or news article.

- How well did trafilatura extract the main content?
Trafilatura did a decent job extracting the main content from the GitHub page. 
It was able to pull out the README content, which includes the description of the project, installation instructions

- Were there any missing or incorrect parts in the extraction?
The extraction was mostly accurate, but there were some formatting issues.

"""

print()
print(todo1_reflection)





- What URL did you choose and why?
https://github.com/openai/whisper
Because in my project, I want to use the whisper model to transcribe audio, 
so i want to see how to use Whisper and what are the features of it. 
I also want to see how well trafilatura can extract the content from a github page, which is different from a typical blog post or news article.

- How well did trafilatura extract the main content?
Trafilatura did a decent job extracting the main content from the GitHub page. 
It was able to pull out the README content, which includes the description of the project, installation instructions

- Were there any missing or incorrect parts in the extraction?
The extraction was mostly accurate, but there were some formatting issues.




---

## Part 2: Markdown Extraction -- Crawl4AI & html2text (2025)

The key insight of modern web scraping for LLMs: **output Markdown, not plain text**. Structured Markdown preserves headings, links, lists, and code blocks -- all of which help LLMs understand document structure.

**Crawl4AI** (50K+ GitHub stars) is the leading tool for this. It uses a headless browser to render JavaScript-heavy pages and outputs clean, LLM-ready Markdown. However, it requires **Python 3.10+**.

For Python 3.9 environments, we use **html2text** as a fallback -- it converts HTML to Markdown without a browser, which works great for static pages like arXiv.

| Tool | Output | JS Support | Python | Best For |
|------|--------|-----------|--------|----------|
| **trafilatura** | Plain text | No | 3.7+ | Bulk extraction, removing boilerplate |
| **html2text** | Markdown | No | 3.6+ | Static pages, lightweight Markdown |
| **Crawl4AI** | Markdown | Yes (browser) | 3.10+ | JS-heavy sites, LLM pipelines |

```
pip install html2text        # Lightweight fallback (works everywhere)
pip install crawl4ai         # Full-featured (requires Python 3.10+)
```

In [28]:
print("=" * 65)
print("Experiment 3: Markdown Extraction vs trafilatura")
print("=" * 65)
print()

# Compare both extractors on the same page
# Uses Crawl4AI if Python 3.10+, otherwise html2text as markdown fallback
comparison_url = "https://arxiv.org/abs/2404.00001"

try:
    comparison = compare_extractors(comparison_url)
    
    print("\n--- trafilatura output (plain text, first 300 chars) ---")
    print(comparison['trafilatura']['text'][:300])
    
    md_result = comparison['crawl4ai']
    print(f"\n--- {md_result.get('method', 'markdown')} output (first 300 chars) ---")
    print(md_result['text'][:300])
    
    print(f"\nKey difference: trafilatura gives plain text ({comparison['trafilatura']['char_count']:,} chars)")
    print(f"Markdown extractor preserves structure ({md_result['char_count']:,} chars)")
except Exception as e:
    print(f"Comparison failed: {e}")
    print("TIP: pip install html2text  (or pip install crawl4ai for Python 3.10+)")

Experiment 3: Markdown Extraction vs trafilatura

EXTRACTOR COMPARISON
URL: https://arxiv.org/abs/2404.00001

[trafilatura] Fetching: https://arxiv.org/abs/2404.00001
  Extracted 2,389 chars in 0.1s
[Crawl4AI] Fetching: https://arxiv.org/abs/2404.00001


[INIT].... → Crawl4AI 0.8.6 

[FETCH]... ↓ https://arxiv.org/abs/2404.00001                                                                     |
✓ | ⏱: 0.56s 

[SCRAPE].. ◆ https://arxiv.org/abs/2404.00001                                                                     |
✓ | ⏱: 0.02s 

[COMPLETE] ● https://arxiv.org/abs/2404.00001                                                                     |
✓ | ⏱: 0.59s 

  Extracted 8,615 chars in 1.5s

--- Comparison ---
  trafilatura:  2,389 chars in 0.1s
  crawl4ai: 8,615 chars in 1.5s

--- trafilatura output (plain text, first 300 chars) ---
Physics > Physics Education
[Submitted on 3 Feb 2024]
Title:Uso de herramientas digitales matemáticas en la Educación Secundaria
View PDF HTML (experimental)Abstract:Information and Community Technologies (ICT) are very present in our society nowadays and particularly in the educative field. In just

--- crawl4ai output (first 300 chars) ---
[Skip to main content](https://arxiv.org/abs/2404.00001#content)
[![Cornell University](https://arxiv.org/static/browse/0.3.4/images/icons/cu/cornell-reduced-white-SMALL.svg)](https://www.cornell.edu/)
[Learn about arXiv becoming an independent nonprofit.](https://tech.cornell.edu/arxiv/)
We gratefu

Key difference: trafilatura gives plain text (2,389 chars)
Markdown extractor preserves structure (8,615 chars)


In [30]:
# TODO 2: Analyze the differences between extractors
#
# Use the LLM to analyze the quality differences between
# trafilatura and Crawl4AI outputs.

print("=" * 65)
print("TODO 2: LLM Analysis of Extraction Quality")
print("=" * 65)
print()

traf_text = blog_result['text'][:500] if blog_result['text'] else 'No extraction'

start = time.time()
response = client.generate(
    prompt=f"""Compare these two web extraction approaches for building LLM pretraining datasets:

1. **trafilatura** (traditional, 2020): Extracts plain text from HTML, removes boilerplate.
   Sample output: {traf_text[:200]}

2. **Crawl4AI** (modern, 2025): Produces LLM-ready Markdown with structure preserved.

For a team building a pretraining dataset:
- Which tool would you recommend for different scenarios?
- What are the trade-offs (speed, quality, features)?
- When would you use one over the other?""",
    system="You are an expert in data engineering for LLM pretraining.",
    max_tokens=1600,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"Response in {elapsed:.1f}s")
    print(format_response(response, verbose=True))
else:
    print(f"Error: {response['error']}")


TODO 2: LLM Analysis of Extraction Quality

⚡ Generating with Gemini...
Response in 11.3s
Model: gemini-3-flash-preview
Tokens: 169 in, 976 out
Stop reason: FinishReason.MAX_TOKENS
As an expert in data engineering for LLMs, the choice between **Trafilatura** and **Crawl4AI** represents a fundamental shift in the industry: moving from **"Text Extraction"** (2020 era) to **"Semantic Content Acquisition"** (2025 era).

For a team building a pretraining dataset, here is the strategic breakdown.

---

### 1. Comparison at a Glance

| Feature | Trafilatura | Crawl4AI |
| :--- | :--- | :--- |
| **Primary Goal** | High-speed boilerplate removal. | LLM-ready structured Markdown. |
| **Engine** | LXML / Regex (Static HTML). | Playwright / Headless Browser (Dynamic). |
| **Speed** | **Extremely Fast** (100s of pages/sec). | **Slower** (Browser overhead). |
| **JS Support** | None (Static only). | Full (React, Vue, SPAs). |
| **Output Quality** | Clean text, loses semantic hierarchy. | Preserves t

In [31]:

todo2_reflection = """


- Which extractor produced better output for your test URLs?
    Crawl4AI produced better output for the test URLs, especially for the arXiv page.
    It preserved the structure and formatting of the content, which is important for LLM pretraining datasets. 
    Trafilatura did a decent job extracting the main content, but it lost some of the formatting and structure,
     which could be valuable for certain applications.   

- For what types of pages would you prefer Crawl4AI over trafilatura?
    I would prefer Crawl4AI for pages where the structure and formatting are important, such as documentation pages, technical blogs, or any content where headings, lists, code blocks, and other markdown features add value. 
    For simpler pages or when I only need the raw text without formatting, trafilatura might be sufficient.         
- How does output format (plain text vs Markdown) affect downstream LLM usage?

    The output format can significantly affect downstream LLM usage. 
    Plain text is simpler and may be easier to process for some applications, but it loses the structural information that can be crucial for understanding the content. 
    Markdown preserves this structure, which can help the LLM better understand the relationships between different parts of the text, such as which sections are headings, which are lists, and where code blocks are. 
    This can lead to better performance in tasks that require understanding of the document structure, such as question answering or summarization. 
"""

print()
print(todo2_reflection)





- Which extractor produced better output for your test URLs?
    Crawl4AI produced better output for the test URLs, especially for the arXiv page.
    It preserved the structure and formatting of the content, which is important for LLM pretraining datasets. 
    Trafilatura did a decent job extracting the main content, but it lost some of the formatting and structure,
     which could be valuable for certain applications.   

- For what types of pages would you prefer Crawl4AI over trafilatura?
    I would prefer Crawl4AI for pages where the structure and formatting are important, such as documentation pages, technical blogs, or any content where headings, lists, code blocks, and other markdown features add value. 
    For simpler pages or when I only need the raw text without formatting, trafilatura might be sufficient.         
- How does output format (plain text vs Markdown) affect downstream LLM usage?

    The output format can significantly affect downstream LLM usage. 
    

---

## Part 3: Building an arXiv Paper Dataset

Let's build a small pretraining dataset by scraping scientific paper abstracts from arXiv. This mirrors what real pretraining pipelines do at scale -- Meta's LLaMA 4 used academic papers as a key data source.

In [34]:
import requests
from xml.etree import ElementTree as ET
base_url = "http://export.arxiv.org/api/query"
params = {
    "search_query": f"all:large language models",
    "start": 0,
    "max_results": 5,
    "sortBy": "submittedDate",
    "sortOrder": "descending",
}

response = requests.get(base_url, params=params, timeout=30)
root = ET.fromstring(response.text)
print(root)

<Element '{http://www.w3.org/2005/Atom}feed' at 0x750764ee49a0>


In [35]:
print("=" * 65)
print("Experiment 4: Scrape arXiv Papers")
print("=" * 65)
print()

papers = scrape_arxiv_abstracts(
    topic="large language models",
    max_results=5,
    save_path=os.path.join(outputs_dir, 'arxiv_papers.json'),
)

Experiment 4: Scrape arXiv Papers

Scraping arXiv: 'large language models' (max 5 papers)

  [1] Posterior Augmented Flow Matching...
      Authors: George Stoica, Sayak Paul, Matthew Wallingford...
      Abstract: Flow matching (FM) trains a time-dependent vector field that transports samples from a simple prior to a complex data di...

  [2] CustomDancer: Customized Dance Recommendation by Text-Dance Retrieval...
      Authors: Yawen Qin, Ke Qiu, Qin Zhang
      Abstract: Dance serves as both a cultural cornerstone and a medium for personal expression, yet the rapid growth of online dance c...

  [3] Reliability, Robustness, and Resilience Modeling for Surveillance System in Adva...
      Authors: Esrat Farhana Dulia, Caleb Adams, Syed Arbab Mohd Shihab...
      Abstract: Ensuring the safe and efficient operation of Advanced Air Mobility (AAM) in low-altitude airspace requires a reliable, r...

  [4] PEARLS: Two Distinct Populations of AGN Hosts Moving Between Star Formation and ...


In [36]:
# TODO 3: Scrape papers on YOUR topic
#
# Choose a topic relevant to your capstone project or interests.
# Scrape 5-10 papers and save the results.

my_topic = "translation"  # e.g., "AI safety", "robotics", "NLP"

print("=" * 65)
print("TODO 3: Custom arXiv Scraping")
print("=" * 65)
print()

my_papers = scrape_arxiv_abstracts(
    topic=my_topic,
    max_results=8,
    save_path=os.path.join(outputs_dir, 'my_arxiv_papers.json'),
)

# Ask Claude to analyze the papers
paper_titles = [p['title'] for p in my_papers]
start = time.time()
response = client.generate(
    prompt=f"""I scraped these {len(my_papers)} papers from arXiv on the topic '{my_topic}':

{chr(10).join(f'{i+1}. {t}' for i, t in enumerate(paper_titles))}

Briefly analyze: What themes do you see? Which 2-3 papers seem most relevant for understanding current trends in this area?""",
    system="You are a research assistant helping analyze academic literature.",
    max_tokens=1400,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"\nAnalysis ({elapsed:.1f}s):")
    print(format_response(response, verbose=False))


TODO 3: Custom arXiv Scraping

Scraping arXiv: 'translation' (max 8 papers)

  [1] Existence of dipoles of Klein-Gordon-Zakharov system...
      Authors: Vicente Alvarez, Amin Esfahani
      Abstract: In this paper, we study the long time behavior of solutions of Klein-Gordon-Zakharov system. We show that there exists a...

  [2] ML-Bench&Guard: Policy-Grounded Multilingual Safety Benchmark and Guardrail for ...
      Authors: Yunhan Zhao, Zhaorun Chen, Xingjun Ma...
      Abstract: As Large Language Models (LLMs) are increasingly deployed in cross-linguistic contexts, ensuring safety in diverse regul...

  [3] EQSANS-CLI: A natural-language, agent-ready command-line tool for small-angle ne...
      Authors: Changwoo Do
      Abstract: Small-angle neutron scattering (SANS) data reduction at user facilities follows a largely repetitive workflow. Runs are ...

  [4] Is Textual Similarity Invariant under Machine Translation? Evidence Based on the...
      Authors: Daria Boratyn, Damian Br

In [37]:

todo3_reflection = """


- What topic did you choose and why?
    I chose the topic of "translation" because my project is focused on building a translation system, 
    and I wanted to see what the latest research trends are in this area.
- Were you surprised by any of the papers found?
    I was surprised to see the mojority of the papers are about NLP--using large language models for translation,
     which shows that this is a very active area of research.
- How could this scraping approach scale to build a real pretraining dataset?
    This scraping approach could be scaled to build a real pretraining dataset by automating the scraping process across multiple topics and sources. 
    We could set up a pipeline that regularly scrapes new papers from arXiv and other repositories, extracts the relevant content, and formats it for LLM pretraining. 
    Additionally, we could use more advanced scraping techniques to handle different formats and structures of academic papers, ensuring that we capture the most useful information for training our models.


"""

print()
print(todo3_reflection)





- What topic did you choose and why?
    I chose the topic of "translation" because my project is focused on building a translation system, 
    and I wanted to see what the latest research trends are in this area.
- Were you surprised by any of the papers found?
    I was surprised to see the mojority of the papers are about NLP--using large language models for translation,
     which shows that this is a very active area of research.
- How could this scraping approach scale to build a real pretraining dataset?
    This scraping approach could be scaled to build a real pretraining dataset by automating the scraping process across multiple topics and sources. 
    We could set up a pipeline that regularly scrapes new papers from arXiv and other repositories, extracts the relevant content, and formats it for LLM pretraining. 
    Additionally, we could use more advanced scraping techniques to handle different formats and structures of academic papers, ensuring that we capture the mo

---

## Summary & Reflection

In [38]:
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[TODO 1 not completed yet]'
_todo2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[TODO 2 not completed yet]'
_todo3 = todo3_reflection.strip() if 'todo3_reflection' in dir() else '[TODO 3 not completed yet]'

full_reflection = f"""
### Part 1 - trafilatura Extraction

{_todo1}

---

### Part 2 - Crawl4AI vs trafilatura

{_todo2}

---

### Part 3 - arXiv Paper Dataset

{_todo3}
"""

reflection_file = append_to_reflection(
    notebook="02",
    section_title="Web Scraping & Text Extraction",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"Reflection saved: {reflection_file}")
print()
tracker.report()

Reflection saved: ../outputs/homework_reflection.md

API COST REPORT
Total API calls:     3
Total input tokens:  563
Total output tokens: 1,631
Total cost:          $0.0262

Last 3 calls:
  1. [11:11:13] 3 -- 169in/23out -- $0.0009
  2. [11:15:10] 3 -- 169in/976out -- $0.0151
  3. [12:08:49] 3 -- 225in/632out -- $0.0102


## Notebook 02 Complete!

**What you accomplished:**
- Extracted clean text from web pages with trafilatura
- Compared traditional vs modern (Crawl4AI) extraction approaches
- Built a mini paper dataset from arXiv

**Key concepts:**
- trafilatura removes HTML boilerplate to extract main content
- Crawl4AI produces LLM-ready Markdown with preserved structure
- arXiv API provides structured access to scientific papers

**Next:** Open **Notebook 03: Document OCR & PDF Extraction**